In [1]:
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA

pd.set_option('display.width', 160)

DATA_PATH = 'Enabel_DATA_LABELS_2026-06-20_1834.csv'
OUT_DIR = 'outputs'

In [2]:
df = pd.read_csv(DATA_PATH, sep=';', dtype=str, engine='python')
print('Dimensions du fichier brut :', df.shape)

Dimensions du fichier brut : (654, 629)


In [3]:
def clean(c):
    """Nettoie un libellé : espaces multiples et retours à la ligne -> un espace."""
    return ' '.join(str(c).split())

clean_cols = [clean(c) for c in df.columns]

def col(prefix, occur=0):
    """Indice de la (occur-ième) colonne dont le libellé commence par `prefix`."""
    hits = [i for i, c in enumerate(clean_cols) if c.startswith(prefix)]
    return hits[occur] if hits else None

def checkbox(prefix, label):
    """Colonne de case à cocher 'prefix ... label' -> indicatrice 1/0."""
    for i, c in enumerate(clean_cols):
        if c.startswith(prefix) and label.lower() in c.lower():
            return (df.iloc[:, i].str.strip() == 'Checked').astype(int)
    return None

In [4]:
base = pd.DataFrame(index=df.index)
base['record_id'] = df.iloc[:, 0]

# Traitement (colonne « 6. Statut de l'enquêtée »)
statut = df.iloc[:, 24].str.strip()
base['T'] = statut.map({"Affilié à l'UDAM": 1, "Non affiliée à l'UDAM": 0})

base['site']     = df.iloc[:, 3].str.strip()
base['district'] = df.iloc[:, col('28.')].str.strip()
base['commune']  = df.iloc[:, 4].str.strip()

ilat = [i for i, c in enumerate(clean_cols) if 'Latitude'  in c][0]
ilon = [i for i, c in enumerate(clean_cols) if 'Longitude' in c][0]
base['gps_lat'] = pd.to_numeric(df.iloc[:, ilat], errors='coerce')
base['gps_lon'] = pd.to_numeric(df.iloc[:, ilon], errors='coerce')

assert base['T'].notna().all(), 'Traitement manquant sur certaines lignes'
print('Répartition du traitement :', base['T'].value_counts().to_dict())
print()
print('Traitement par site (non-recouvrement de Fatick) :')
print(pd.crosstab(base['site'], base['T'], margins=True))

Répartition du traitement : {1: 329, 0: 325}

Traitement par site (non-recouvrement de Fatick) :
T              0    1  All
site                      
Fatick       169    0  169
Foundiougne  121  179  300
Koungheul     35  150  185
All          325  329  654


In [5]:
X = pd.DataFrame(index=df.index)
X['record_id'] = base['record_id'].values

# Démographie
X['age']    = pd.to_numeric(df.iloc[:, col('346.')], errors='coerce')
X['mariee'] = (df.iloc[:, col('347.')].str.strip() == 'Mariées').astype(int)

# Statut socio-économique (ordinal)
rev = {'Inférieur 50000': 1, '50000 - 100 000': 2, '100 001 - 200 000': 3, 'Supérieur 200 000': 4}
X['revenu_ord'] = df.iloc[:, col('348.')].str.strip().map(rev)
ins = {'Sans instruction': 0, 'Primaire': 1, 'Secondaire': 2, 'Supérieur': 3}
X['instruction_ord'] = df.iloc[:, col('349.')].str.strip().map(ins)

# Occupation agricole / élevage (fréquente dans la zone)
agri = checkbox('350.', 'Cultivateur'); elev = checkbox('350.', 'Éleveur')
X['agri_elevage'] = ((agri.fillna(0) + elev.fillna(0)) > 0).astype(int)

# Histoire obstétricale (critères d'appariement = pré-traitement)
X['nb_grossesses']            = pd.to_numeric(df.iloc[:, col('30.')], errors='coerce')
X['nb_enfants']               = pd.to_numeric(df.iloc[:, col('31.')], errors='coerce')
X['antecedent_complication']  = checkbox('32.', 'Complication')
X['accouchement_dystocique']  = checkbox('33.', 'Dystocique')

# Cadre de vie (servira aussi à l'indice de richesse)
X['proprietaire'] = (df.iloc[:, col('353.')].str.strip() == 'Oui').astype(int)
murs = df.iloc[:, col('354.')].str.strip()
X['murs_finis'] = murs.str.startswith('Matériaux fi').astype(int)
eau_rob = checkbox('355.', 'Robinet dans la maison')
eau_for = checkbox('355.', 'Forage rural')
eau_pp  = checkbox('355.', 'Puits protégé')
X['eau_amelioree'] = ((eau_rob.fillna(0) + eau_for.fillna(0) + eau_pp.fillna(0)) > 0).astype(int)

print('Confondeurs construits :', [c for c in X.columns if c != 'record_id'])

Confondeurs construits : ['age', 'mariee', 'revenu_ord', 'instruction_ord', 'agri_elevage', 'nb_grossesses', 'nb_enfants', 'antecedent_complication', 'accouchement_dystocique', 'proprietaire', 'murs_finis', 'eau_amelioree']


In [6]:
T = base['T'].values

def smd(x):
    x = pd.to_numeric(x, errors='coerce')
    a, b = x[T == 1], x[T == 0]
    sp = np.sqrt((a.var(ddof=1) + b.var(ddof=1)) / 2)
    return np.nan if sp == 0 else (a.mean() - b.mean()) / sp

conf_cols = [c for c in X.columns if c != 'record_id']
bal = pd.DataFrame({
    'moy_cas':    [round(pd.to_numeric(X[c], errors='coerce')[T == 1].mean(), 2) for c in conf_cols],
    'moy_temoin': [round(pd.to_numeric(X[c], errors='coerce')[T == 0].mean(), 2) for c in conf_cols],
    'SMD':        [round(smd(X[c]), 3) for c in conf_cols],
}, index=conf_cols)
print(bal.to_string())
print()
print('Confondeurs déséquilibrés (|SMD| > 0.10) :',
      bal.index[bal['SMD'].abs() > 0.10].tolist())

                         moy_cas  moy_temoin    SMD
age                        24.91       26.34 -0.231
mariee                      0.98        0.97  0.039
revenu_ord                  2.01        2.47 -0.397
instruction_ord             0.57        1.13 -0.623
agri_elevage                0.02        0.01  0.077
nb_grossesses               3.22        3.03  0.097
nb_enfants                  2.99        2.91  0.046
antecedent_complication     0.10        0.06  0.158
accouchement_dystocique     0.10        0.08  0.082
proprietaire                0.54        0.57 -0.069
murs_finis                  0.84        0.94 -0.292
eau_amelioree               0.89        0.93 -0.147

Confondeurs déséquilibrés (|SMD| > 0.10) : ['age', 'revenu_ord', 'instruction_ord', 'antecedent_complication', 'murs_finis', 'eau_amelioree']


In [7]:
assets = ['Électricité', 'Radio', 'TV', 'Téléphone portable', 'Réfrigérateur', 'Bicyclette',
          'Motocyclette', 'Charrette', 'Voiture/camion', 'Panneau solaire', 'Tables', 'Chaise', 'Lit']
A = pd.DataFrame(index=df.index)
for a in assets:
    for i, c in enumerate(clean_cols):
        if 'biens ou les équipements' in c and a.lower() in c.lower():
            A[a] = (df.iloc[:, i].str.strip() == 'Checked').astype(int)
            break
# Ajout des marqueurs de cadre de vie
A['Murs finis']    = X['murs_finis'].values
A['Propriétaire']  = X['proprietaire'].values
A['Eau améliorée'] = X['eau_amelioree'].values

# On écarte les indicateurs quasi constants (aucun pouvoir discriminant)
prev = A.mean()
keep = prev[(prev > 0.02) & (prev < 0.98)].index.tolist()
Ak = A[keep]

# ACP sur indicateurs standardisés ; 1re composante = indice de richesse
Z = (Ak - Ak.mean()) / Ak.std(ddof=0)
pca = PCA(n_components=5, random_state=0).fit(Z)
pc1 = pca.transform(Z)[:, 0]
if np.corrcoef(pc1, Ak.sum(axis=1))[0, 1] < 0:   # orientation : + d'actifs => indice + élevé
    pc1 = -pc1

X['wealth_index']    = pc1
X['wealth_quintile'] = pd.qcut(pc1, 5, labels=[1, 2, 3, 4, 5]).astype(int)

print('Variance expliquée par la 1re composante : %.1f%%' % (100 * pca.explained_variance_ratio_[0]))
print('Corrélation indice / revenu déclaré : %.2f' %
      pd.Series(pc1).corr(pd.to_numeric(X['revenu_ord'], errors='coerce')))
print('Indice moyen - cas : %.2f | témoin : %.2f | SMD : %.3f' %
      (pc1[T == 1].mean(), pc1[T == 0].mean(), smd(pd.Series(pc1))))

Variance expliquée par la 1re composante : 22.8%
Corrélation indice / revenu déclaré : 0.31
Indice moyen - cas : -0.44 | témoin : 0.45 | SMD : -0.476


In [8]:
def num(i):
    return pd.to_numeric(df.iloc[:, i].astype(str).str.replace(' ', '').str.replace(',', '.'),
                         errors='coerce')

Y = pd.DataFrame(index=df.index)
Y['record_id'] = base['record_id'].values

# (A) Dépense directe totale : colonnes de montant = libellé contenant 'Combien' ET 'payé'
cost_idx = [i for i, c in enumerate(clean_cols) if ('Combien' in c) and ('payé' in c)]
COST = pd.DataFrame({i: num(i) for i in cost_idx})
Y['oop_total'] = COST.fillna(0).sum(axis=1)
print('Colonnes de montant agrégées :', len(cost_idx))

# (B) Dépense EXCESSIVE — méthode de Tukey (valeur aberrante)
#     Excessive si dépense > Q3 + k*(Q3 - Q1). Seuil calé sur les NON AFFILIÉES (non traitées)
ref = Y['oop_total'][T == 0]
q1, q3 = ref.quantile(0.25), ref.quantile(0.75)
iqr = q3 - q1
print('Distribution de référence (non affiliées) : Q1=%.0f  Q3=%.0f  IQR=%.0f' % (q1, q3, iqr))
for k in [0.5, 1.0, 1.5]:
    fence = q3 + k * iqr
    Y['exces_k%02d' % int(k * 10)] = (Y['oop_total'] > fence).astype(int)
    print('  seuil dépense excessive k=%.1f : %8.0f FCFA  ->  %.1f%% des femmes au total' %
          (k, fence, 100 * (Y['oop_total'] > fence).mean()))

# (C) Score de cascade de soins
cpn        = num(col('48.'))
step_cpn4  = (cpn >= 4).astype(int)
step_accou = (df.iloc[:, col('190.')].str.strip() == 'Structure de soins').astype(int)
step_pnat  = (df.iloc[:, col('280.')].str.strip() == 'Oui').astype(int)
step_vacc  = (df.iloc[:, col('330.')].str.strip() == 'Oui').astype(int)
Y['cascade_score']    = step_cpn4 + step_accou + step_pnat + step_vacc
Y['cascade_complete'] = (Y['cascade_score'] == 4).astype(int)

# Aperçu BRUT (non ajusté) - à titre indicatif
print('OOP médian - cas : %.0f | témoin : %.0f' %
      (Y['oop_total'][T == 1].median(), Y['oop_total'][T == 0].median()))
print('Dépense excessive (k=1,5) - cas : %.1f%% | témoin : %.1f%%' %
      (100 * Y['exces_k15'][T == 1].mean(), 100 * Y['exces_k15'][T == 0].mean()))
print('Cascade moyenne - cas : %.2f | témoin : %.2f' %
      (Y['cascade_score'][T == 1].mean(), Y['cascade_score'][T == 0].mean()))

Colonnes de montant agrégées : 53
Distribution de référence (non affiliées) : Q1=49400  Q3=92000  IQR=42600
  seuil dépense excessive k=0.5 :   113300 FCFA  ->  8.9% des femmes au total
  seuil dépense excessive k=1.0 :   134600 FCFA  ->  3.4% des femmes au total
  seuil dépense excessive k=1.5 :   155900 FCFA  ->  2.0% des femmes au total
OOP médian - cas : 39500 | témoin : 68500
Dépense excessive (k=1,5) - cas : 1.5% | témoin : 2.5%
Cascade moyenne - cas : 3.42 | témoin : 3.36


In [9]:
CONF_KEEP = ['age', 'mariee', 'revenu_ord', 'instruction_ord', 'agri_elevage',
             'nb_grossesses', 'nb_enfants', 'antecedent_complication', 'accouchement_dystocique']

adata = base[['record_id', 'T', 'site', 'district', 'commune', 'gps_lat', 'gps_lon']].copy()
adata = adata.merge(X[['record_id'] + CONF_KEEP + ['wealth_index', 'wealth_quintile']], on='record_id')
adata = adata.merge(Y, on='record_id')

# Imputation simple du manquant réel (faible)
cont = ['age', 'nb_grossesses', 'nb_enfants']
cats = ['mariee', 'revenu_ord', 'instruction_ord', 'agri_elevage',
        'antecedent_complication', 'accouchement_dystocique']
for v in cont:
    if adata[v].isna().any():
        adata[v] = adata[v].fillna(adata[v].median())
for v in cats:
    if adata[v].isna().any():
        adata[v] = adata[v].fillna(adata[v].mode(dropna=True).iloc[0])

resid = {v: int(adata[v].isna().sum()) for v in CONF_KEEP + ['wealth_index']}
print('Manquant résiduel sur confondeurs :', {k: v for k, v in resid.items() if v > 0} or 'aucun')
print('Dimensions du jeu analytique :', adata.shape)

Manquant résiduel sur confondeurs : aucun
Dimensions du jeu analytique : (654, 24)


In [10]:
import os
os.makedirs(OUT_DIR, exist_ok=True)
adata.to_csv(f'{OUT_DIR}/analytic_dataset.csv', index=False, encoding='utf-8-sig')
adata.to_parquet(f'{OUT_DIR}/analytic_dataset.parquet')
print('Jeu analytique sauvegardé -> {}/analytic_dataset.csv (.parquet)'.format(OUT_DIR))
adata.head()

Jeu analytique sauvegardé -> outputs/analytic_dataset.csv (.parquet)


,record_id,T,site,district,commune,gps_lat,gps_lon,age,mariee,revenu_ord,...,antecedent_complication,accouchement_dystocique,wealth_index,wealth_quintile,oop_total,exces_k05,exces_k10,exces_k15,cascade_score,cascade_complete
0,1,0,Foundiougne,DS de Sokone,Karang,13.612654,-16.428747,25.0,1,3.0,...,0,0,-0.741281,2,27000.0,0,0,0,4,1
1,2,1,Foundiougne,DS de Sokone,Karang,13.595237,-16.424218,20.0,1,1.0,...,0,0,-0.062043,3,41000.0,0,0,0,4,1
2,3,1,Foundiougne,DS de Passy,Diagane mbarka,13.997288,-16.293113,34.0,1,1.0,...,0,0,-2.515492,1,22000.0,0,0,0,3,0
3,4,0,Foundiougne,DS de Passy,Diagane mbarka,13.998246,-16.293898,20.0,1,1.0,...,0,0,-2.034129,1,33500.0,0,0,0,4,1
4,5,1,Foundiougne,DS de Passy,Diagane mbarka,13.997175,-16.293104,23.0,1,2.0,...,0,0,-0.942654,2,25000.0,0,0,0,4,1
